# Confronto Robyn vs Meridian - Parte 1: Robyn sul dataset demo

Fit di **Robyn** su `dt_simulated_weekly` (dataset demo ufficiale di Robyn:
208 settimane nazionali, 5 canali paid, 1 organico, 2 variabili di contesto).

**Prima di iniziare:** Runtime -> Cambia tipo di runtime -> **R** (CPU va bene).
Tempo totale stimato: ~60-90 min (installazione ~15-20 min, run ~40-60 min).

Output per il confronto (da scaricare a fine run):
- `dt_simulated_weekly.csv` (backup del dataset, riusabile nel notebook Meridian)
- `robyn_metrics.csv` (R2 e NRMSE su train/val/test dei modelli candidati)
- `robyn_roas.csv` (ROAS e decomposizione per canale del modello selezionato)
- cartella `robyn_output/` (onepager e grafici)

## 1. Setup (~15-20 min): Robyn da CRAN + nevergrad

In [ ]:
install.packages("Robyn")   # CRAN, versione >= 3.12
system("pip install nevergrad")
library(reticulate)
use_python(Sys.which("python3"), required = TRUE)
py_config()  # verifica che nevergrad sia visibile a reticulate
stopifnot(py_module_available("nevergrad"))

## 2. Dataset demo

Esporta subito il CSV: lo stesso file va usato (o riscaricato da GitHub) nel
notebook Meridian, cosi' i due modelli vedono esattamente gli stessi dati.

In [ ]:
library(Robyn)
data("dt_simulated_weekly")
data("dt_prophet_holidays")
write.csv(dt_simulated_weekly, "dt_simulated_weekly.csv", row.names = FALSE)
cat("Settimane:", nrow(dt_simulated_weekly), "\n")
print(range(dt_simulated_weekly$DATE))
head(dt_simulated_weekly)

## 3. Specifica del modello (come demo.R ufficiale)

Scelte allineate col notebook Meridian (vedi `protocollo_confronto.md`):
- finestra di modellazione = **intero dataset** (nessun warmup asimmetrico)
- `train_size` fissato a **0.8** -> validation = penultimo 10%, **test = ultimo 10%**
  (~21 settimane). Il test e' la finestra di confronto con Meridian.
- variabili esposizione dove disponibili: `facebook_I`, `search_clicks_P`

In [ ]:
InputCollect <- robyn_inputs(
  dt_input = dt_simulated_weekly,
  dt_holidays = dt_prophet_holidays,
  date_var = "DATE",
  dep_var = "revenue",
  dep_var_type = "revenue",
  prophet_vars = c("trend", "season", "holiday"),
  prophet_country = "DE",
  context_vars = c("competitor_sales_B", "events"),
  paid_media_spends = c("tv_S", "ooh_S", "print_S", "facebook_S", "search_S"),
  paid_media_vars = c("tv_S", "ooh_S", "print_S", "facebook_I", "search_clicks_P"),
  organic_vars = "newsletter",
  window_start = min(dt_simulated_weekly$DATE),
  window_end = max(dt_simulated_weekly$DATE),
  adstock = "geometric"
)

In [ ]:
# range degli iperparametri: valori del demo ufficiale.
# NB: per i canali con variabile di esposizione i nomi usano
# l'esposizione (facebook_I, search_clicks_P), non la spesa.
# Verifica sempre con:
# hyper_names(adstock = InputCollect$adstock, all_media = InputCollect$all_media)
hyperparameters <- list(
  tv_S_alphas = c(0.5, 3), tv_S_gammas = c(0.3, 1), tv_S_thetas = c(0.3, 0.8),
  ooh_S_alphas = c(0.5, 3), ooh_S_gammas = c(0.3, 1), ooh_S_thetas = c(0.1, 0.4),
  print_S_alphas = c(0.5, 3), print_S_gammas = c(0.3, 1), print_S_thetas = c(0.1, 0.4),
  facebook_I_alphas = c(0.5, 3), facebook_I_gammas = c(0.3, 1), facebook_I_thetas = c(0, 0.3),
  search_clicks_P_alphas = c(0.5, 3), search_clicks_P_gammas = c(0.3, 1), search_clicks_P_thetas = c(0, 0.3),
  newsletter_alphas = c(0.5, 3), newsletter_gammas = c(0.3, 1), newsletter_thetas = c(0.1, 0.4),
  # intervallo strettissimo (estremi identici scatenano un bug noto di
  # hyper_collector): test = ultimo ~10%, circa 21 settimane
  train_size = c(0.79, 0.81)
)
InputCollect <- robyn_inputs(InputCollect = InputCollect,
                             hyperparameters = hyperparameters)

## 4. Run (~40-60 min su CPU Colab)

In [ ]:
OutputModels <- robyn_run(
  InputCollect = InputCollect,
  iterations = 2000,
  trials = 5,
  ts_validation = TRUE,
  add_penalty_factor = FALSE,
  cores = max(1, parallel::detectCores() - 1),
  seed = 42
)

OutputCollect <- robyn_outputs(
  InputCollect, OutputModels,
  pareto_fronts = "auto",
  csv_out = "pareto",
  clusters = TRUE,
  export = TRUE,
  plot_folder = "robyn_output",
  plot_pareto = TRUE
)

## 5. Selezione del modello ed export metriche

Tra i vincitori dei cluster si seleziona qui quello col miglior NRMSE sul **test**.
Nota per la tesi: la selezione tra i candidati di Pareto e' un grado di
liberta' dell'analista che Meridian non ha (un solo posterior) -- punto da
discutere nel confronto. Guarda comunque gli onepager in `robyn_output/`
prima di accettare la scelta automatica.

In [ ]:
rh <- OutputCollect$resultHypParam
candidati <- if (!is.null(OutputCollect$clusters)) {
  OutputCollect$clusters$models$solID
} else rh$solID
cand <- rh[rh$solID %in% candidati, ]
cand <- cand[order(cand$nrmse_test), ]
sel <- cand$solID[1]
cat("Modello selezionato:", sel, "\n")

cols_m <- intersect(c("solID", "rsq_train", "rsq_val", "rsq_test",
                      "nrmse_train", "nrmse_val", "nrmse_test",
                      "decomp.rssd", "mape", "train_size"), names(cand))
metriche <- cand[, cols_m]
write.csv(metriche, "robyn_metrics.csv", row.names = FALSE)
metriche

In [ ]:
xd <- OutputCollect$xDecompAgg
cols_r <- intersect(c("rn", "total_spend", "mean_spend", "xDecompAgg",
                      "xDecompPerc", "spend_share", "effect_share",
                      "roi_mean", "roi_total"), names(xd))
roas <- xd[xd$solID == sel & xd$rn %in% c(InputCollect$paid_media_spends, InputCollect$paid_media_vars), cols_r]
write.csv(roas, "robyn_roas.csv", row.names = FALSE)
roas

# onepager del modello selezionato
invisible(robyn_onepagers(InputCollect, OutputCollect, select_model = sel,
                          export = TRUE))

## 6. Scarica i risultati

Il runtime R di Colab non ha `files.download`: crea lo zip e scaricalo dal
**pannello file a sinistra** (icona cartella -> tasto destro -> Download).

In [ ]:
zip("robyn_risultati.zip",
    files = c("robyn_metrics.csv", "robyn_roas.csv", "dt_simulated_weekly.csv"))
cat("Creato robyn_risultati.zip: scaricalo dal pannello file a sinistra.\n",
    "Scarica anche la cartella robyn_output/ (onepager).\n")

### (opzionale) Stabilita' tra run
Per il criterio di stabilita' del protocollo: rilancia le sezioni 4-5 con
`seed = 43` e `seed = 44` e confronta ROAS e NRMSE test tra i tre run.